In [1]:
# Install KaggleHub and dependencies
!pip install kagglehub pandas sqlalchemy openpyxl --quiet

import pandas as pd
import sqlite3
import kagglehub
from kagglehub import KaggleDatasetAdapter


In [2]:
# Download dataset folder
dataset_path = kagglehub.dataset_download("raghu07/ipl-data-till-2017")
print("Dataset downloaded to:", dataset_path)


100%|██████████| 1.60M/1.60M [00:00<00:00, 107MB/s]

Extracting files...
Dataset downloaded to: /root/.cache/kagglehub/datasets/raghu07/ipl-data-till-2017/versions/4


In [3]:
# Define file paths (dataset contains Match.csv, Ball_By_Ball.csv, etc.)
match_fp = dataset_path + "/Match.csv"
ball_fp = dataset_path + "/Ball_By_Ball.csv"
player_fp = dataset_path + "/Player.csv"
team_fp = dataset_path + "/Team.csv"
player_match_fp = dataset_path + "/Player_match.csv"

# Read into pandas
matches = pd.read_csv(match_fp, encoding="latin1")
balls = pd.read_csv(ball_fp, encoding="latin1")
players = pd.read_csv(player_fp, encoding="latin1")
teams = pd.read_csv(team_fp, encoding="latin1")
player_match = pd.read_csv(player_match_fp, encoding="latin1")

print(" Data Loaded")
print("Matches:", matches.shape, " Balls:", balls.shape, " Players:", players.shape)


 Data Loaded
Matches: (637, 17)  Balls: (150451, 48)  Players: (497, 7)


In [4]:
matches_2017 = matches[matches["Season_Year"] == 2017]
balls_2017 = balls[balls["Season"] == 2017]
player_match_2017 = player_match[player_match["Season_year"] == 2017]

print("Matches 2017:", matches_2017.shape)


Matches 2017: (60, 17)


In [5]:
# Aggregate runs scored by striker
batsman_runs = balls_2017.groupby("Striker").agg(Runs=('Runs_Scored','sum')).reset_index()

# Join with Player to get names
batsman_runs = batsman_runs.merge(players[['Player_Id','Player_Name']],
                                  left_on="Striker", right_on="Player_Id", how="left")

top_batsmen = batsman_runs[['Player_Name','Runs']].sort_values("Runs", ascending=False).head(10)

print(" Top Batsmen 2017 by Runs:")
print(top_batsmen)


 Top Batsmen 2017 by Runs:
     Player_Name  Runs
39     DA Warner   641
12     G Gambhir   498
13      S Dhawan   479
62     SPD Smith   472
6       SK Raina   442
113      HM Amla   420
26     MK Pandey   396
4       PA Patel   395
45    KA Pollard   395
136  RA Tripathi   391


In [6]:
# Drop matches without winners
matches_2017 = matches_2017.dropna(subset=["match_winner"])

# Wins per team
team_wins = matches_2017['match_winner'].value_counts().reset_index()
team_wins.columns = ['Team', 'Wins']

print(" Teams with Most Wins 2017:")
print(team_wins)


 Teams with Most Wins 2017:
                          Team  Wins
0               Mumbai Indians    11
1      Rising Pune Supergiants    10
2        Kolkata Knight Riders     9
3          Sunrisers Hyderabad     8
4              Kings XI Punjab     7
5             Delhi Daredevils     6
6                Gujarat Lions     4
7  Royal Challengers Bangalore     3
8                    abandoned     1
9                         tied     1


In [7]:
# Count awards directly by player name
mvp_awards = matches_2017['ManOfMach'].value_counts().reset_index()
mvp_awards.columns = ['Player_Name','Awards']

# Top MVPs
top_mvps = mvp_awards[['Player_Name','Awards']].sort_values("Awards", ascending=False).head(10)

print(" Top MVPs 2017:")
print(top_mvps)


 Top MVPs 2017:
       Player_Name  Awards
0        BA Stokes       3
1  NM Coulter-Nile       3
2           AJ Tye       2
3           N Rana       2
4       RV Uthappa       2
5   Sandeep Sharma       2
6        KH Pandya       2
7      Rashid Khan       2
8       JD Unadkat       2
9        KM Jadhav       1


In [8]:
conn = sqlite3.connect("ipl2017_etl.db")

top_batsmen.to_sql("Top_Batsmen_2017", conn, if_exists="replace", index=False)
team_wins.to_sql("Team_Wins_2017", conn, if_exists="replace", index=False)
top_mvps.to_sql("Top_MVPs_2017", conn, if_exists="replace", index=False)

print(" ETL Completed. Results saved to SQLite database (ipl2017_etl.db)")


 ETL Completed. Results saved to SQLite database (ipl2017_etl.db)


In [9]:
with pd.ExcelWriter("IPL2017_ETL_Insights.xlsx") as writer:
    top_batsmen.to_excel(writer, sheet_name="TopBatsmen", index=False)
    team_wins.to_excel(writer, sheet_name="TeamWins", index=False)
    top_mvps.to_excel(writer, sheet_name="TopMVPs", index=False)

from google.colab import files
files.download("IPL2017_ETL_Insights.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>